# Marcas Diarias - Market Data

Este notebook muestra cómo traer las marcas diarias del servidor (Supabase) usando el `MarketDataLoader`.

**Fuentes de datos:**
- `sofr_swap_curve` — Curva SOFR par swap (Eris Futures)
- `banrep_series_value_v2` — Depósitos IBR (BanRep)
- `ibr_2y/5y/10y` — Swaps IBR (fuentes directas)
- `cop_fwd_points` — Forward points USD/COP (FXEmpire)
- `us_reference_rates` — SOFR overnight

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'), override=True)

from pricing.data.market_data import MarketDataLoader
import pandas as pd

pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.6f}'.format)

loader = MarketDataLoader()
print('MarketDataLoader inicializado OK')

# ── Fecha de las marcas ──
# None = última fecha disponible, o poner fecha específica: '2026-03-02'
FECHA_MARCAS = None

if FECHA_MARCAS:
    print(f'Fecha seleccionada: {FECHA_MARCAS}')
else:
    print('Fecha: última disponible')

## 1. Curva SOFR (22 tenores)

In [ ]:
sofr_df = loader.fetch_sofr_curve(target_date=FECHA_MARCAS)
fecha_sofr = FECHA_MARCAS or loader._latest_date("sofr_swap_curve")
print(f'Fecha de la marca: {fecha_sofr}')
print(f'Tenores: {len(sofr_df)}')
print()
sofr_df['tenor_label'] = sofr_df['tenor_months'].apply(
    lambda m: f'{m}M' if m < 12 else f'{m//12}Y'
)
sofr_df

## 2. Quotes IBR (depósitos + swaps)

In [ ]:
ibr_quotes = loader.fetch_ibr_quotes(target_date=FECHA_MARCAS)
print(f'Fecha IBR: {FECHA_MARCAS or "última disponible"}')
print('IBR Quotes (rates en %):')
print()
ibr_df = pd.DataFrame([
    {'tenor': k, 'rate_pct': v[0]} for k, v in ibr_quotes.items()
])
ibr_df

## 3. USD/COP Spot y Forward Points

In [ ]:
spot = loader.fetch_usdcop_spot(target_date=FECHA_MARCAS)
cop_fwd = loader.fetch_cop_forwards(target_date=FECHA_MARCAS)
fecha_fwd = FECHA_MARCAS or loader._latest_date("cop_fwd_points")

print(f'SET-ICAP spot:   {spot:,.2f} COP/USD')
print(f'Fecha FXEmpire:  {fecha_fwd}')
print()

# Lo que almacenamos (fwd_points en pips ×10000)
print('── Almacenado en cop_fwd_points ──')
display(cop_fwd[['tenor', 'tenor_months', 'fwd_points']])

print()

# Lo que entra en la curva NDF
fwd_curve = cop_fwd[cop_fwd.tenor_months > 0].copy()
fwd_curve['F_market'] = spot + fwd_curve['fwd_points'] / 10_000
fwd_curve['fwd_pts_cop'] = fwd_curve['fwd_points'] / 10_000

print('── Curva NDF: F_market = SET-ICAP spot + fwd_points/10000 ──')
display(fwd_curve[['tenor', 'tenor_months', 'fwd_pts_cop', 'F_market']].reset_index(drop=True))

## 4. SOFR Overnight (referencia)

In [ ]:
sofr_spot = loader.fetch_sofr_spot(target_date=FECHA_MARCAS)
print(f'SOFR overnight rate: {sofr_spot*100:.4f}%' if sofr_spot else 'No disponible')

## 5. Catálogo TES (bonos activos)

In [ ]:
tes_df = loader.fetch_tes_bond_info()
print(f'Bonos TES en catálogo: {len(tes_df)}')
tes_df

## 6. Construir Curvas (CurveManager)

Ahora construimos las curvas IBR y SOFR con `CurveManager.build_all()` y vemos el status.

In [ ]:
from pricing.curves.curve_manager import CurveManager
import QuantLib as ql

cm = CurveManager()

# Construir curvas con la data ya cargada (respeta FECHA_MARCAS)
if ibr_quotes:
    cm.build_ibr_curve(ibr_quotes)
if not sofr_df.empty:
    cm.build_sofr_curve(sofr_df)
if spot:
    cm.set_fx_spot(spot)

print(f'Fecha marcas: {FECHA_MARCAS or "última disponible"}')
print(f'Valuation date: {cm.valuation_date}')
print(f'FX Spot: {cm.fx_spot}')
print(f'IBR: {"OK" if cm.ibr_curve else "NO"}')
print(f'SOFR: {"OK" if cm.sofr_curve else "NO"}')

In [ ]:
# Nodos de la curva IBR
status = cm.status()
print('=== IBR Curve Nodes (%) ===')
for k, v in status['ibr']['nodes'].items():
    print(f'  {k:12s}: {v:.4f}%')

print()
print('=== SOFR Curve Nodes (%) ===')
for k, v in status['sofr']['nodes'].items():
    print(f'  {k:>4s}M: {v:.4f}%')

## 7. Discount Factors y Zero Rates (verificación)

In [ ]:
# Tabla de discount factors y zero rates para tenores estándar
tenors = [1, 3, 6, 12, 24, 36, 60, 84, 120, 180, 240]
eval_date = cm.valuation_date

rows = []
for m in tenors:
    dt = eval_date + ql.Period(m, ql.Months)
    label = f'{m}M' if m < 12 else f'{m//12}Y'
    rows.append({
        'tenor': label,
        'ibr_df': cm.ibr_discount(dt),
        'ibr_zero': cm.ibr_zero_rate(dt) * 100,
        'sofr_df': cm.sofr_discount(dt),
        'sofr_zero': cm.sofr_zero_rate(dt) * 100,
    })

df_curves = pd.DataFrame(rows)
df_curves

In [ ]:
from pricing.curves.ndf_curve import build_ndf_curve

if not cop_fwd.empty and spot and cm.sofr_curve:
    ndf_curve, fwd_pts_dict = build_ndf_curve(cop_fwd, spot, cm.sofr_handle, cm.valuation_date)
    ndf_handle = ql.YieldTermStructureHandle(ndf_curve)

    print(f'SET-ICAP spot: {spot:,.2f} COP/USD')
    print()

    rows = []
    for months, fwd_pts_cop in sorted(fwd_pts_dict.items()):
        dt    = cm.valuation_date + ql.Period(months, ql.Months)
        df_n  = ndf_handle.discount(dt)
        df_u  = cm.sofr_discount(dt)
        f_mkt = spot + fwd_pts_cop
        deval = ((f_mkt / spot) ** (12 / months) - 1) * 100
        label = f'{months}M' if months < 12 else f'{months//12}Y'
        rows.append({
            'tenor':       label,
            'F_market':    round(f_mkt, 2),
            'fwd_pts_cop': round(fwd_pts_cop, 2),
            'deval_ea_%':  round(deval, 2),
            'ndf_df':      round(df_n, 6),
            'sofr_df':     round(df_u, 6),
        })

    ndf_df = pd.DataFrame(rows)
    print(ndf_df.to_string(index=False))
else:
    print('Faltan datos para construir la curva NDF')

## 8. Curva NDF (forwards USD/COP implícitos)

Construida con **Método B**: `F_market = spot_SET-ICAP + fwd_points / 10_000`

- `spot` → `currency_hour` (SET-ICAP, intraday)  
- `fwd_points` → `cop_fwd_points` (FXEmpire, pip convention ×10000)  
- Formula: `DF_COP(T) = spot × DF_USD(T) / F_market(T)`